# Training notebook — five-optimizer benchmark (IJISTA)

This notebook contains the complete training and evaluation protocol for the manuscript **“Comparative Analysis of Optimizers in CNNs for Hematological Image Classification”**.

**Scope**

- Optimizers: **SGD with momentum, RMSprop, Adam, AdamW and RAdam**.
- Architectures: **ResNet18, EfficientNetV2-S and MobileNetV3-Large**.
- Datasets: **PBC, Naturalize 2K-PBC and Raabin-WBC**.
- Seeds: **10 per configuration** (`7, 42, 123, 456, 784, 999, 1337, 2026, 3407, 8888`).
- Total: **3 datasets × 3 architectures × 5 optimizers × 10 seeds = 450 runs**.

All models are trained from scratch. The reported metrics are final-epoch test metrics. The notebook writes one CSV per scenario and the aggregate summary and paired Wilcoxon/Holm files.

> **Note on reproducibility.** The CSV files in this repository are the reported results. Re-running the notebook on a different GPU, CUDA/cuDNN version or library version can produce small numerical differences even with the same seeds.


## 1. Environment and imports

The original runs used Python 3.13.9 and PyTorch 2.14.0 in Google Colab (NVIDIA T4 or L4). Install the Kaggle CLI only if you want the notebook to download the datasets.


In [ ]:
# If you are in Colab and need the Kaggle CLI, uncomment:
# !pip -q install kaggle

import os
import sys
import time
import random
import shutil
import glob
import json
import math

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset, random_split, WeightedRandomSampler
from torchvision.datasets import ImageFolder
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score
from sklearn.preprocessing import label_binarize

print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__)
print("torchvision :", torchvision.__version__)
print("CUDA        :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))


## 2. Global configuration

The five optimizers and their fixed hyperparameters are defined once here. No learning-rate scheduler and no early stopping are used.


In [ ]:
SEEDS = [7, 42, 123, 456, 784, 999, 1337, 2026, 3407, 8888]

NUM_EPOCHS = 80
BATCH_SIZE = 64
NUM_WORKERS = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OPTIMIZERS = ["SGD", "RMSprop", "Adam", "AdamW", "RAdam"]

HYPERPARAMS = {
    "SGD":     {"lr": 0.01,  "momentum": 0.9, "weight_decay": 0.0},
    "RMSprop": {"lr": 0.001, "alpha": 0.99, "eps": 1e-8, "weight_decay": 0.0},
    "Adam":    {"lr": 0.001, "betas": (0.9, 0.999), "eps": 1e-8, "weight_decay": 0.0},
    "AdamW":   {"lr": 0.001, "betas": (0.9, 0.999), "eps": 1e-8, "weight_decay": 0.01},
    "RAdam":   {"lr": 0.001, "betas": (0.9, 0.999), "eps": 1e-8, "weight_decay": 0.0},
}

print("Device:", DEVICE)
for name in OPTIMIZERS:
    print(f"{name:8s} -> {HYPERPARAMS[name]}")


## 3. Datasets, paths and class labels

The notebook expects the following layout under `DATA_ROOT`:

```
data/
├── pbc/full_dataset/{basophil, eosinophil, erythroblast, ig, lymphocyte, monocyte, neutrophil, platelet}/
├── naturalize/Naturalize 2K-PBC Dataset/{Train, Validate, Test}/
└── raabin/full_dataset/{Basophil, Eosinophil, Lymphocyte, Monocyte, Neutrophil}/
```

Set `PREPARE_DATA = True` to download and consolidate the public Kaggle datasets automatically. This requires a valid `~/.kaggle/kaggle.json`.


In [ ]:
DATA_ROOT = "./data"
RESULTS_DIR = "./resultados"
PREPARE_DATA = False   # set to True to download/consolidate from Kaggle

os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

DATASETS = {
    "PBC": {
        "num_classes": 8,
        "img_size": 64,
        "sampler_factor": 2,
        "split": "seeded_random_70_15_15",
        "classes": [
            "basophil", "eosinophil", "erythroblast", "ig",
            "lymphocyte", "monocyte", "neutrophil", "platelet",
        ],
    },
    "Naturalize2K": {
        "num_classes": 11,
        "img_size": 96,
        "sampler_factor": 3,
        "split": "official_train_validate_test",
        "classes": [
            "Basophil", "Banded Neutrophil", "Eosinophil", "Erythroblast",
            "Lymphocyte", "Meta-myelocyte", "Monocyte", "Myelocyte",
            "Platelet", "Pro-myelocyte", "Segmented Neutrophil",
        ],
    },
    "Raabin-WBC": {
        "num_classes": 5,
        "img_size": 128,
        "sampler_factor": 3,
        "split": "seeded_random_70_15_15",
        "classes": ["Basophil", "Eosinophil", "Lymphocyte", "Monocyte", "Neutrophil"],
    },
}

ARCHITECTURES = ["ResNet18", "EfficientNetV2", "MobileNetV3"]

for key, cfg in DATASETS.items():
    print(f"{key:12s} | classes={cfg['num_classes']:2d} | {cfg['img_size']}x{cfg['img_size']} | {cfg['split']}")


## 4. Reproducibility and dataset preparation

`set_seed` fixes Python, NumPy and PyTorch RNGs. For each seed, the data partition is created once and reused by the five optimizers; the RNG is reset before each model is initialized.


In [ ]:
def set_seed(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def _run_kaggle(dataset_slug, target_dir):
    os.makedirs(target_dir, exist_ok=True)
    cmd = f"kaggle datasets download -d {dataset_slug} -p {target_dir} --unzip"
    print(cmd)
    if PREPARE_DATA:
        status = os.system(cmd)
        if status != 0:
            raise RuntimeError(f"Kaggle download failed: {cmd}")
    else:
        raise FileNotFoundError(
            f"Dataset not found in {target_dir}. Set PREPARE_DATA=True or place the data manually."
        )


def prepare_pbc():
    root = os.path.join(DATA_ROOT, "pbc")
    full = os.path.join(root, "full_dataset")
    categories = DATASETS["PBC"]["classes"]
    if os.path.isdir(full) and all(os.path.isdir(os.path.join(full, c)) for c in categories):
        return full

    _run_kaggle("unclesamulus/blood-cells-image-dataset", root)
    os.makedirs(full, exist_ok=True)
    for c in categories:
        os.makedirs(os.path.join(full, c), exist_ok=True)

    extensions = (".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".JPG", ".JPEG", ".PNG", ".BMP", ".TIFF")
    images = []
    for ext in extensions:
        images.extend(glob.glob(os.path.join(root, "**", f"*{ext}"), recursive=True))

    for path in images:
        parent = os.path.basename(os.path.dirname(path)).lower()
        if parent == "erythrocyte":
            parent = "erythroblast"
        if parent in categories:
            shutil.copy2(path, os.path.join(full, parent))
    return full


def prepare_naturalize():
    root = os.path.join(DATA_ROOT, "naturalize")
    base = os.path.join(root, "Naturalize 2K-PBC Dataset")
    expected = ["Train", "Validate", "Test"]
    if os.path.isdir(base) and all(os.path.isdir(os.path.join(base, s)) for s in expected):
        return base

    _run_kaggle("mohamadabouali1/blood-cells-dataset-11-classes-26534-images", root)
    if not os.path.isdir(base):
        raise FileNotFoundError(
            f"Expected folder not found: {base}. Check the extracted Kaggle folder name."
        )
    return base


def prepare_raabin():
    root = os.path.join(DATA_ROOT, "raabin")
    full = os.path.join(root, "full_dataset")
    categories = DATASETS["Raabin-WBC"]["classes"]
    if os.path.isdir(full) and all(os.path.isdir(os.path.join(full, c)) for c in categories):
        return full

    _run_kaggle("raabindata/raabin-wbc", root)
    os.makedirs(full, exist_ok=True)
    for c in categories:
        os.makedirs(os.path.join(full, c), exist_ok=True)

    source_dirs = [
        os.path.join(root, "Train", "Train"),
        os.path.join(root, "TestA", "TestA"),
        os.path.join(root, "TestB", "Test-B"),
    ]
    extensions = (".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG")
    for c in categories:
        for src in source_dirs:
            src_dir = os.path.join(src, c)
            if not os.path.isdir(src_dir):
                continue
            for path in glob.glob(os.path.join(src_dir, "*")):
                if path.lower().endswith(extensions):
                    shutil.copy2(path, os.path.join(full, c))
    return full


## 5. Transforms

The augmentation recipe is dataset-specific and follows the original experiments. Validation and test use resize + ImageNet normalization only.


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def build_transforms(dataset, img_size):
    normalize = transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)

    if dataset == "PBC":
        train_tf = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=30),
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.8, 1.2)),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
            transforms.RandomResizedCrop(size=img_size, scale=(0.7, 1.0)),
            transforms.ToTensor(),
            normalize,
        ])
    elif dataset == "Naturalize2K":
        train_tf = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=25),
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.8, 1.2)),
            transforms.RandomResizedCrop(size=img_size, scale=(0.8, 1.0)),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
            transforms.RandomGrayscale(p=0.1),
            transforms.ToTensor(),
            normalize,
        ])
    elif dataset == "Raabin-WBC":
        train_tf = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(degrees=15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
            transforms.ToTensor(),
            normalize,
        ])
    else:
        raise ValueError(f"Unknown dataset: {dataset}")

    eval_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        normalize,
    ])
    return train_tf, eval_tf


## 6. Dataset and DataLoader builders

For PBC and Raabin-WBC, the 70/15/15 split is generated with `torch.Generator().manual_seed(seed)`. Naturalize 2K-PBC uses its official `Train` / `Validate` / `Test` folders. The weighted sampler is built only from the training partition.


In [ ]:
class TransformSubset(Dataset):
    """Apply a transform to a torch.utils.data.Subset."""

    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform is not None:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)


def build_datasets(dataset, seed):
    cfg = DATASETS[dataset]
    img_size = cfg["img_size"]
    train_tf, eval_tf = build_transforms(dataset, img_size)

    if dataset == "Naturalize2K":
        base = prepare_naturalize()
        train_ds = ImageFolder(os.path.join(base, "Train"), transform=train_tf)
        train_eval_ds = ImageFolder(os.path.join(base, "Train"), transform=eval_tf)
        val_ds = ImageFolder(os.path.join(base, "Validate"), transform=eval_tf)
        test_ds = ImageFolder(os.path.join(base, "Test"), transform=eval_tf)
    else:
        root = prepare_pbc() if dataset == "PBC" else prepare_raabin()
        full = ImageFolder(root)
        n = len(full)
        n_train = int(0.70 * n)
        n_val = int(0.15 * n)
        n_test = n - n_train - n_val
        train_sub, val_sub, test_sub = random_split(
            full, [n_train, n_val, n_test],
            generator=torch.Generator().manual_seed(seed),
        )
        train_ds = TransformSubset(train_sub, transform=train_tf)
        train_eval_ds = TransformSubset(train_sub, transform=eval_tf)
        val_ds = TransformSubset(val_sub, transform=eval_tf)
        test_ds = TransformSubset(test_sub, transform=eval_tf)

    return train_ds, train_eval_ds, val_ds, test_ds


def _train_labels(train_ds):
    if isinstance(train_ds, TransformSubset):
        base = train_ds.subset.dataset
        indices = train_ds.subset.indices
        return [base.targets[i] for i in indices]
    return [label for _, label in train_ds.samples]


def make_loaders(train_ds, train_eval_ds, val_ds, test_ds, dataset):
    cfg = DATASETS[dataset]
    num_classes = cfg["num_classes"]
    labels = _train_labels(train_ds)
    class_counts = np.bincount(labels, minlength=num_classes)
    class_weights = 1.0 / (class_counts + 1e-8)
    sample_weights = [class_weights[label] for label in labels]
    num_samples = int(class_counts.min()) * num_classes * cfg["sampler_factor"]

    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=num_samples,
        replacement=True,
    )

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, sampler=sampler,
        num_workers=NUM_WORKERS, pin_memory=True,
    )
    train_eval_loader = DataLoader(
        train_eval_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
    )
    test_loader = DataLoader(
        test_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
    )

    print(f"  Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}")
    print(f"  Sampler samples/epoch: {num_samples:,} | minority class: {class_counts.min():,}")
    return train_loader, train_eval_loader, val_loader, test_loader


## 7. Architectures

All backbones use random initialization (`weights=None`). The final classification layer is replaced by dropout (`p = 0.5`) and a linear layer with the target number of classes.


In [ ]:
def build_model(arch, num_classes):
    if arch == "ResNet18":
        model = models.resnet18(weights=None)
        in_features = model.fc.in_features
        model.fc = nn.Sequential(nn.Dropout(p=0.5), nn.Linear(in_features, num_classes))
    elif arch == "EfficientNetV2":
        model = models.efficientnet_v2_s(weights=None)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Sequential(nn.Dropout(p=0.5), nn.Linear(in_features, num_classes))
    elif arch == "MobileNetV3":
        model = models.mobilenet_v3_large(weights=None)
        in_features = model.classifier[3].in_features
        model.classifier[3] = nn.Sequential(nn.Dropout(p=0.5), nn.Linear(in_features, num_classes))
    else:
        raise ValueError(f"Unknown architecture: {arch}")
    return model


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

for arch in ARCHITECTURES:
    m = build_model(arch, 8)
    print(f"{arch:14s} trainable parameters: {count_parameters(m)/1e6:.2f} M")
    del m


## 8. Optimizer factory

Only the five optimizers in the manuscript are constructed. The exact hyperparameters are the ones in Section 2.


In [ ]:
def build_optimizer(name, params):
    hp = HYPERPARAMS[name]
    if name == "SGD":
        return optim.SGD(
            params, lr=hp["lr"], momentum=hp["momentum"],
            weight_decay=hp["weight_decay"],
        )
    if name == "RMSprop":
        return optim.RMSprop(
            params, lr=hp["lr"], alpha=hp["alpha"], eps=hp["eps"],
            weight_decay=hp["weight_decay"],
        )
    if name == "Adam":
        return optim.Adam(
            params, lr=hp["lr"], betas=hp["betas"], eps=hp["eps"],
            weight_decay=hp["weight_decay"],
        )
    if name == "AdamW":
        return optim.AdamW(
            params, lr=hp["lr"], betas=hp["betas"], eps=hp["eps"],
            weight_decay=hp["weight_decay"],
        )
    if name == "RAdam":
        return optim.RAdam(
            params, lr=hp["lr"], betas=hp["betas"], eps=hp["eps"],
            weight_decay=hp["weight_decay"],
        )
    raise ValueError(f"Unknown optimizer: {name}")


## 9. Training and evaluation

Each run trains for 80 epochs with cross-entropy loss, gradient clipping (`max_norm = 1.0`) and no scheduler. Validation is evaluated after every epoch to track the best epoch. The reported test metrics are computed once, after the final epoch.


In [ ]:
def evaluate(model, loader, criterion, num_classes, compute_auc=True):
    model.eval()
    total_loss = 0.0
    y_true, y_pred, y_scores = [], [], []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * inputs.size(0)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(outputs.argmax(dim=1).cpu().numpy())
            if compute_auc:
                y_scores.append(torch.softmax(outputs, dim=1).cpu().numpy())

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    auc = float("nan")
    if compute_auc:
        y_scores = np.vstack(y_scores)
        y_bin = label_binarize(y_true, classes=list(range(num_classes)))
        try:
            auc = roc_auc_score(y_bin, y_scores, average="macro", multi_class="ovr")
        except Exception:
            auc = float("nan")

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(y_true, y_pred),
        "auc": auc,
        "f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }


def train_one(dataset, arch, optimizer_name, seed,
              train_loader, train_eval_loader, val_loader, test_loader):
    cfg = DATASETS[dataset]
    num_classes = cfg["num_classes"]
    img_size = cfg["img_size"]

    set_seed(seed)
    model = build_model(arch, num_classes).to(DEVICE)
    optimizer = build_optimizer(optimizer_name, model.parameters())
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    best_epoch = 0
    start_time = time.time()

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        train_metrics = evaluate(
            model, train_eval_loader, criterion, num_classes, compute_auc=False
        )
        val_metrics = evaluate(
            model, val_loader, criterion, num_classes, compute_auc=False
        )
        if val_metrics["accuracy"] > best_val_acc:
            best_val_acc = val_metrics["accuracy"]
            best_epoch = epoch

        if epoch == 1 or epoch % 10 == 0 or epoch == NUM_EPOCHS:
            print(
                f"    epoch {epoch:3d}/{NUM_EPOCHS} | "
                f"train loss {train_metrics['loss']:.4f} acc {train_metrics['accuracy']*100:.2f}% | "
                f"val loss {val_metrics['loss']:.4f} acc {val_metrics['accuracy']*100:.2f}%"
            )

    test_metrics = evaluate(
        model, test_loader, criterion, num_classes, compute_auc=True
    )
    elapsed = time.time() - start_time

    row = {
        "architecture": arch,
        "dataset": dataset,
        "resolution": f"{img_size}x{img_size}",
        "seed": seed,
        "optimizer": optimizer_name,
        "accuracy_pct": test_metrics["accuracy"] * 100.0,
        "loss": test_metrics["loss"],
        "auc": test_metrics["auc"],
        "f1_pct": test_metrics["f1"] * 100.0,
        "recall_pct": test_metrics["recall"] * 100.0,
        "time_s": elapsed,
    }
    print(
        f"  {optimizer_name:8s} seed {seed:5d} -> "
        f"acc {row['accuracy_pct']:.2f}% | loss {row['loss']:.4f} | "
        f"AUC {row['auc']:.5f} | F1 {row['f1_pct']:.2f}% | {elapsed/60:.1f} min"
    )
    return row


## 10. Scenario runner

A scenario is one dataset–architecture pair. For each seed, the split and the loaders are built once, and then the five optimizers are trained with the same partition. Rows are written incrementally so that an interrupted run keeps the completed results; the final file is sorted by optimizer and seed.


In [ ]:
def run_scenario(dataset, arch, seeds=SEEDS, optimizers=OPTIMIZERS, out_csv=None):
    if dataset not in DATASETS:
        raise ValueError(f"Unknown dataset: {dataset}")
    if arch not in ARCHITECTURES:
        raise ValueError(f"Unknown architecture: {arch}")

    if out_csv is not None:
        os.makedirs(os.path.dirname(out_csv) or ".", exist_ok=True)
        if os.path.exists(out_csv):
            os.replace(out_csv, out_csv + ".bak")

    rows = []
    for seed in seeds:
        print(f"\n=== {dataset} | {arch} | seed {seed} ===")
        train_ds, train_eval_ds, val_ds, test_ds = build_datasets(dataset, seed)
        train_loader, train_eval_loader, val_loader, test_loader = make_loaders(
            train_ds, train_eval_ds, val_ds, test_ds, dataset
        )

        for optimizer_name in optimizers:
            row = train_one(
                dataset=dataset,
                arch=arch,
                optimizer_name=optimizer_name,
                seed=seed,
                train_loader=train_loader,
                train_eval_loader=train_eval_loader,
                val_loader=val_loader,
                test_loader=test_loader,
            )
            rows.append(row)
            if out_csv is not None:
                pd.DataFrame([row]).to_csv(
                    out_csv,
                    mode="a",
                    header=not os.path.exists(out_csv),
                    index=False,
                )

    result = pd.DataFrame(rows).sort_values(["optimizer", "seed"]).reset_index(drop=True)
    if out_csv is not None:
        result.to_csv(out_csv, index=False)
    return result


## 11. Run the benchmark

Set `RUN_ALL = True` to execute the nine scenarios (450 runs). For a quick check, keep `RUN_ALL = False` and edit `SELECTED_SCENARIOS`. A full run is computationally expensive; each scenario produces its own CSV.


In [ ]:
ALL_SCENARIOS = [
    (dataset, arch)
    for dataset in ["PBC", "Naturalize2K", "Raabin-WBC"]
    for arch in ARCHITECTURES
]

RUN_ALL = False
SELECTED_SCENARIOS = [
    ("PBC", "ResNet18"),
]

scenarios = ALL_SCENARIOS if RUN_ALL else SELECTED_SCENARIOS
all_rows = []

for dataset, arch in scenarios:
    safe_name = f"{dataset}_{arch}".replace(" ", "_")
    out_csv = os.path.join(RESULTS_DIR, f"{safe_name}_resultados_por_semilla.csv")
    print(f"\n########## {dataset} | {arch} ##########")
    scenario_df = run_scenario(dataset, arch, out_csv=out_csv)
    all_rows.append(scenario_df)

if all_rows:
    combined = pd.concat(all_rows, ignore_index=True)
    combined = combined.sort_values(
        ["dataset", "architecture", "optimizer", "seed"]
    ).reset_index(drop=True)
    combined.to_csv(os.path.join(RESULTS_DIR, "semillas_5_optimizadores.csv"), index=False)
    print(f"\nSaved {len(combined)} rows.")


## 12. Aggregate summary

The summary uses sample standard deviations (`ddof = 1`), matching the manuscript. It can be run after any subset of scenarios; missing scenarios are simply absent.


In [ ]:
def build_summary(seed_csv):
    df = pd.read_csv(seed_csv)
    group_cols = ["dataset", "architecture", "optimizer"]
    rows = []
    for keys, g in df.groupby(group_cols):
        dataset, arch, opt = keys
        rows.append({
            "dataset": dataset,
            "architecture": arch,
            "optimizer": opt,
            "n_seeds": len(g),
            "accuracy_pct_mean": g["accuracy_pct"].mean(),
            "accuracy_pct_sd_sample": g["accuracy_pct"].std(ddof=1),
            "loss_mean": g["loss"].mean(),
            "loss_sd_sample": g["loss"].std(ddof=1),
            "auc_mean": g["auc"].mean(),
            "auc_sd_sample": g["auc"].std(ddof=1),
            "f1_pct_mean": g["f1_pct"].mean(),
            "f1_pct_sd_sample": g["f1_pct"].std(ddof=1),
            "recall_pct_mean": g["recall_pct"].mean(),
            "recall_pct_sd_sample": g["recall_pct"].std(ddof=1),
            "time_s_mean": g["time_s"].mean(),
            "time_s_sd_sample": g["time_s"].std(ddof=1),
        })
    return pd.DataFrame(rows)


seed_csv = os.path.join(RESULTS_DIR, "semillas_5_optimizadores.csv")
if os.path.exists(seed_csv):
    summary = build_summary(seed_csv)
    summary.to_csv(os.path.join(RESULTS_DIR, "resumen_5_optimizadores.csv"), index=False)
    display(summary.head(10))
else:
    print("Run Section 11 first.")


## 13. Paired Wilcoxon tests with Holm correction

For each scenario, all 10 pairs of optimizers are compared with a two-sided paired Wilcoxon signed-rank test on the 10 seed-level accuracy values. Holm correction is applied within each scenario at `alpha = 0.05`. This reproduces the columns in `wilcoxon_pareado_holm_5_optimizadores.csv`.


In [ ]:
from scipy.stats import wilcoxon


def paired_wilcoxon_holm(seed_csv):
    df = pd.read_csv(seed_csv)
    out = []

    for (dataset, arch), g in df.groupby(["dataset", "architecture"]):
        pivot = g.pivot(index="seed", columns="optimizer", values="accuracy_pct")
        optimizers = [o for o in ["Adam", "AdamW", "RAdam", "SGD", "RMSprop"] if o in pivot.columns]
        if len(optimizers) < 2:
            continue
        pairs = []

        for i in range(len(optimizers)):
            for j in range(i + 1, len(optimizers)):
                a, b = optimizers[i], optimizers[j]
                x = pivot[a].to_numpy()
                y = pivot[b].to_numpy()
                diff = x - y

                wins_a = int((diff > 0).sum())
                ties = int((diff == 0).sum())
                wins_b = int((diff < 0).sum())

                nonzero = diff[diff != 0]
                abs_diff = np.abs(nonzero)
                ranks = pd.Series(abs_diff).rank(method="average").to_numpy()
                w_plus = float(ranks[nonzero > 0].sum()) if len(nonzero) else 0.0
                w_minus = float(ranks[nonzero < 0].sum()) if len(nonzero) else 0.0

                try:
                    _, p_value = wilcoxon(x, y, alternative="two-sided", method="auto")
                except Exception:
                    _, p_value = wilcoxon(x, y, alternative="two-sided", method="approx")

                pairs.append({
                    "dataset": dataset,
                    "architecture": arch,
                    "optimizer_a": a,
                    "optimizer_b": b,
                    "mean_delta_a_minus_b_pp": float(diff.mean()),
                    "wins_a": wins_a,
                    "ties": ties,
                    "wins_b": wins_b,
                    "W_plus": w_plus,
                    "W_minus": w_minus,
                    "p_two_sided_exact": float(p_value),
                })

        p_values = np.array([p["p_two_sided_exact"] for p in pairs])
        order = np.argsort(p_values)
        m = len(p_values)
        adjusted = np.empty(m, dtype=float)
        previous = 0.0
        for rank, idx in enumerate(order):
            value = min(1.0, (m - rank) * p_values[idx])
            value = max(value, previous)
            adjusted[idx] = value
            previous = value

        for p, adj in zip(pairs, adjusted):
            p["p_holm_within_scenario"] = float(adj)
            p["significant_alpha_0_05"] = bool(adj < 0.05)
            out.append(p)

    return pd.DataFrame(out)


if os.path.exists(seed_csv):
    wilcoxon_df = paired_wilcoxon_holm(seed_csv)
    wilcoxon_df.to_csv(
        os.path.join(RESULTS_DIR, "wilcoxon_pareado_holm_5_optimizadores.csv"),
        index=False,
    )
    display(wilcoxon_df.head(10))
else:
    print("Run Section 11 first.")


## 14. Notes and limitations

- The reported metrics are final-epoch test metrics; there is no early stopping.
- Each dataset is evaluated at one resolution: 64 × 64 for PBC, 96 × 96 for Naturalize 2K-PBC and 128 × 128 for Raabin-WBC.
- PBC and Raabin-WBC use seeded random image-level splits; no patient-level grouping was available.
- Adam uses zero weight decay and AdamW uses 0.01, so their contrast is not a pure test of decoupled weight decay.
- The results characterize this fixed-rate, from-scratch protocol and do not establish clinical superiority or generalization to other tasks.
